# Selected Tool Argument Generation

## tl;dr

- small-baseline: historical 13/24, this run stage1 13/24, two-stage final 11/24.
- medium-candidate: historical 16/24, this run stage1 16/24, two-stage final 24/24.
- prompt-variant: historical 14/24, this run stage1 15/24, two-stage final 20/24.

Local diagnostic evidence only. Defaults and the official Gate are unchanged.

## Context & Methods

Question: does a second, selected-schema-only model call preserve explicit values? Stage1 retains the legacy full-call prompt, but its arguments are discarded. Stage2 returns an argument object; the final Tool envelope is assembled without value repair.

### Key Assumptions

Frozen cohort: 12 cases x 3 entries x 2 trials. Historical baseline was measured earlier. Challenge: 6 newly authored cases x 3 entries x 2 Tool orders x 2 variants, seed 42. AB/BA order alternates. These are not independently reviewed or statistically independent samples. Latency is local wall time, not an SLA; cache and host load are uncontrolled. Fault traces replay one output three times and use deterministic local fixtures.

## Data

### 1. Verify Sources

Source: `artifacts/reference-workload/tool-two-stage-diagnostic-v1.json`. SHA-256: `f122e1e6cb38c349b00d8d35769f279a21786a16139f1ab4834eff36558f0ca8`. The audit reads only local files, makes no model calls and performs no application DB writes. Source hashes, journal equality, trial completeness, request parity, usage totals, unchanged arguments and raw-output exact matches are checked independently of the application scorer.

In [1]:
import hashlib, json, sys
from pathlib import Path
root = Path.cwd()
while not (root / 'reference_workload/runtime_matrix.json').exists():
    assert root != root.parent, 'repository root not found'
    root = root.parent
source = root / 'artifacts/reference-workload/tool-two-stage-diagnostic-v1.json'
assert hashlib.sha256(source.read_bytes()).hexdigest() == 'f122e1e6cb38c349b00d8d35769f279a21786a16139f1ab4834eff36558f0ca8'
auditor = root / 'tools/audit_tool_two_stage.py'
assert hashlib.sha256(auditor.read_bytes()).hexdigest() == '1106c0d0871e651355f5fa310780799bf1a16e2e78324f57c51fb27891acaa9c'
sys.path.insert(0, str(root / 'tools'))
from audit_tool_two_stage import audit
result = audit(root, source)
print({key: result[key] for key in ('assessment', 'observations', 'http_calls', 'trace_count')})

{'assessment': 'share_with_caveats_local_diagnostic_only', 'observations': 144, 'http_calls': 252, 'trace_count': 432}


## Results

### 2. Compare Exact Calls and Costs

Exact means selected Tool and the entire argument object match explicitly requested values. All planned observations remain in the denominator. Token totals use complete HTTP usage only.

In [2]:
from IPython.display import Markdown, display
headers = ['entry', 'cohort', 'variant', 'n', 'stage1_exact', 'exact', 'prompt_tokens', 'completion_tokens', 'median_latency_ms']
table = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
for row in result['summary']:
    table.append('| ' + ' | '.join(str(round(row[key], 1)) if isinstance(row[key], float) else str(row[key]) for key in headers) + ' |')
display(Markdown('\n'.join(table)))

| entry | cohort | variant | n | stage1_exact | exact | prompt_tokens | completion_tokens | median_latency_ms |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| small-baseline | frozen | candidate | 24 | 13 | 11 | 26979 | 1023 | 1476.2 |
| small-baseline | challenge | baseline | 12 | 6 | 6 | 10424 | 347 | 1111.4 |
| small-baseline | challenge | candidate | 12 | 6 | 9 | 13686 | 578 | 1537.9 |
| medium-candidate | frozen | candidate | 24 | 16 | 24 | 27460 | 1127 | 2489.5 |
| medium-candidate | challenge | baseline | 12 | 7 | 7 | 10424 | 475 | 2022.2 |
| medium-candidate | challenge | candidate | 12 | 6 | 9 | 14086 | 664 | 2724.8 |
| prompt-variant | frozen | candidate | 24 | 15 | 20 | 29020 | 1033 | 2445.6 |
| prompt-variant | challenge | baseline | 12 | 6 | 6 | 10760 | 468 | 1946.7 |
| prompt-variant | challenge | candidate | 12 | 7 | 9 | 14422 | 625 | 2530.5 |

### 3. Check Selection Drift and Regressions

Within-candidate transitions compare that run's first response and final response. This avoids attributing repeated-run selection changes to stage2. Order-change counts have 18 case-entry pairs per variant.

In [3]:
print(json.dumps({key: result[key] for key in (
    'within_candidate_exact_transitions', 'frozen_stage1_selection_changed_vs_historical',
    'challenge_pair_stage1_selection_changes', 'challenge_order_exact_changes',
    'finish_reasons')}, indent=2))
print('Non-exact observations:', len(result['failures']))

{
  "within_candidate_exact_transitions": {
    "frozen/0->1": 15,
    "frozen/0->0": 13,
    "frozen/1->1": 40,
    "frozen/1->0": 4,
    "challenge/1->1": 16,
    "challenge/0->1": 11,
    "challenge/0->0": 6,
    "challenge/1->0": 3
  },
  "frozen_stage1_selection_changed_vs_historical": 0,
  "challenge_pair_stage1_selection_changes": 0,
  "challenge_order_exact_changes": {
    "baseline": 7,
    "candidate": 3
  },
  "finish_reasons": {
    "stop": 252
  }
}
Non-exact observations: 43


## Takeaways

Retain this as an opt-in diagnostic candidate, not a default replacement. Use the per-entry results, regressions, and extra token cost together. Fresh independent requests, real Tool content/authorization validation and human review remain necessary before promotion. Native-output schema validity must not be conflated with the candidate's assembled-envelope validity.

In [4]:
target = root / 'docs/reports/2026-09-09_tool_two_stage/audit.json'
target.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Audit saved:', target.relative_to(root))

Audit saved: docs\reports\2026-09-09_tool_two_stage\audit.json
